In [1]:
# install HuggingFace 'datasets'
!pip install datasets

In [1]:
# load wikitext from datasets
# using wikitext2 raw version

from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [2]:
# total rows and non-empty rows dont match here
# need to do some cleaning
count_nonempty = 0
count_total = 0

for entry in dataset["train"]:
    count_total += 1
    if entry["text"].strip() != "":
        count_nonempty += 1

print("Total rows:", count_total)
print("Non-empty rows:", count_nonempty)

Total rows: 36718
Non-empty rows: 23767


In [3]:
# remove blank lines
dataset = dataset.filter(lambda x: x["text"].strip() != "")

In [4]:
# checking what data looks like
print(dataset["train"][0])

{'text': ' = Valkyria Chronicles III = \n'}


In [5]:
# now the row numbers match
count_nonempty = 0
count_total = 0

for entry in dataset["train"]:
    count_total += 1
    if entry["text"].strip() != "":
        count_nonempty += 1

print("Total rows:", count_total)
print("Non-empty rows:", count_nonempty)

Total rows: 23767
Non-empty rows: 23767


In [6]:
# PRE-PROCESSING
# 're' for regex
# this function extracts and cleans text
# note to self: wikitext2 is about 2M words
# this counts about 1.75M
import re

def preprocess(text):
    text = text.lower()
    return re.findall(r'\b\w+\b', text)

# building the corpus
all_words = []

for entry in dataset["train"]:
    text = entry["text"]
    if text.strip():   # simple safe guard
        all_words.extend(preprocess(text))

print("Total words:", len(all_words))

Total words: 1750956


In [7]:
#count unique words and most frequent words
from collections import Counter

word_freq = Counter(all_words)

print("Unique words:", len(word_freq))
print(word_freq.most_common(10))

Unique words: 65867
[('the', 130771), ('of', 57032), ('and', 50738), ('in', 45019), ('to', 39522), ('a', 36567), ('was', 21008), ('on', 15141), ('as', 15058), ('s', 14982)]


In [8]:
# function to do prefix based matching for suggestions
# return top 5 suggestions
def get_suggestions(prefix, word_freq, k=5):
    prefix = prefix.lower()

    matches = [word for word in word_freq if word.startswith(prefix)]

    ranked = sorted(matches, key=lambda w: word_freq[w], reverse=True)

    return ranked[:k]

In [9]:
# testing autocomplete for "comp" and "auto"
print(get_suggestions("comp", word_freq))
print(get_suggestions("auto", word_freq))

['company', 'completed', 'complete', 'companies', 'competition']
['autobiography', 'automobile', 'automatic', 'auto', 'autonomy']


In [10]:
# testing autocomplete for "a" and "b"
print(get_suggestions("a", word_freq))
print(get_suggestions("b", word_freq))

['and', 'a', 'as', 'at', 'an']
['by', 'be', 'but', 'been', 'between']


In [11]:
def autocomplete(prefix, word_freq, k=5):
    prefix = prefix.lower()

    # Find matching words
    matches = [word for word in word_freq if word.startswith(prefix)]

    # Rank by frequency
    matches = sorted(matches, key=lambda w: word_freq[w], reverse=True)

    return matches[:k]

In [12]:
# test the autocomplete function
print(autocomplete("comp", word_freq))
print(autocomplete("auto", word_freq))
print(autocomplete("data", word_freq))

['company', 'completed', 'complete', 'companies', 'competition']
['autobiography', 'automobile', 'automatic', 'auto', 'autonomy']
['data', 'database', 'datadyne', 'databases', 'dataflow']


In [13]:
# simulate someone typing
prefix = "c"
for i in range(1, len("computer") + 1):
    current = "computer"[:i]
    print(f"Input: {current}")
    print("Suggestions:", autocomplete(current, word_freq))
    print()

Input: c
Suggestions: ['city', 'can', 'century', 'called', 'could']

Input: co
Suggestions: ['could', 'company', 'considered', 'continued', 'county']

Input: com
Suggestions: ['company', 'common', 'completed', 'community', 'come']

Input: comp
Suggestions: ['company', 'completed', 'complete', 'companies', 'competition']

Input: compu
Suggestions: ['computer', 'computers', 'compulsory', 'computing', 'computational']

Input: comput
Suggestions: ['computer', 'computers', 'computing', 'computational', 'computation']

Input: compute
Suggestions: ['computer', 'computers', 'computed', 'computerized', 'compute']

Input: computer
Suggestions: ['computer', 'computers', 'computerized', 'computerised']



In [14]:
# was the CORRECT word suggested?
def top_k_accuracy(test_words, word_freq, k=5):
    correct = 0
    total = 0

    for word in test_words:
        if len(word) < 3:
            continue

        prefix = word[:3]  # simulate user typing first 3 letters
        suggestions = autocomplete(prefix, word_freq, k)

        if word in suggestions:
            correct += 1

        total += 1

    return correct / total if total > 0 else 0

In [19]:
#sample = list(word_freq.keys())[:1000]
import random

sample = random.sample(list(word_freq.keys()), 1000)

print("Top-k accuracy:", top_k_accuracy(sample, word_freq, k=5))

Top-k accuracy: 0.2631578947368421


In [16]:
# how many characters did the user avoid having to type?
def keystroke_savings(word, word_freq, k=5):
    for i in range(1, len(word)):
        prefix = word[:i]
        suggestions = autocomplete(prefix, word_freq, k)

        if word in suggestions:
            # user stops typing here
            return len(word) - i

    return 0

In [17]:
#
def average_keystroke_savings(words, word_freq, k=5):
    total = 0

    for word in words:
        if len(word) < 3:
            continue
        total += keystroke_savings(word, word_freq, k)

    return total / len(words)

In [22]:
sample = random.sample(list(word_freq.keys()), 1000)

print("Avg keystroke savings:", average_keystroke_savings(sample, word_freq, k=5))

Avg keystroke savings: 3.108


In [23]:
# EXPERIMENT
def keystroke_savings_real(word, word_counts, k=5):
    for i in range(1, len(word)):
        prefix = word[:i]
        suggestions = autocomplete(prefix, word_counts, k)

        if word in suggestions:
            # user would stop typing here
            return len(word) - i

    # never predicted
    return 0

In [24]:
def average_keystroke_savings_real(sample, word_counts, k=5):
    total = 0

    for word in sample:
        if len(word) < 3:
            continue
        total += keystroke_savings_real(word, word_counts, k)

    return total / len(sample)

In [25]:
sample = random.sample(list(word_freq.keys()), 1000)

print("Avg keystroke savings:", average_keystroke_savings_real(sample, word_freq, k=5))

Avg keystroke savings: 3.072
